# 01 Dataset Audit

Purpose:
- validate the raw dataset structure
- count participants and recordings by class
- detect missing files, inconsistent naming, and leakage risks

Primary questions:
- How many participants exist in each class?
- How many recordings exist per participant?
- Are there missing or duplicate files?
- Can we reliably identify each patient for patient-level modeling?


In [9]:
from pathlib import Path
# Load config (YAML) from backend/config/config.yaml — fail fast if missing
try:
    import yaml
except Exception:
    raise ImportError('PyYAML is required to read backend/config/config.yaml. Install with `pip install pyyaml`')
cfg_path = Path('backend') / 'config' / 'config.yaml'
cfg_path = cfg_path.resolve()
if not cfg_path.exists():
    # common notebook cwd is backend/notebooks; try backend/config/config.yaml relative to that
    alt = Path.cwd().resolve().parent / 'config' / 'config.yaml'
    if alt.exists():
        cfg_path = alt
    else:
        raise FileNotFoundError(f'Config file not found: {cfg_path}')
with cfg_path.open('r') as f:
    cfg = yaml.safe_load(f)
project_root = cfg_path.parents[2]
DATA_DIR = (project_root / cfg.get('data_dir', 'backend/data')).resolve()
DATASET_SUBDIR = cfg.get('dataset_subdir', 'audio_lanzhou_2015')
METADATA_WORKBOOK = cfg.get('metadata_workbook', 'subjects_information_audio_lanzhou_2015.xlsx')
print(f'project_root: {project_root}')
print(f'Using data dir: {DATA_DIR}')
print(f'Dataset subdir: {DATASET_SUBDIR}')
print(f'Metadata workbook: {METADATA_WORKBOOK}')
DATA_DIR

project_root: /Users/vishalkarda/Documents/Projects/mendx.ai
Using data dir: /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data
Dataset subdir: audio_lanzhou_2015
Metadata workbook: subjects_information_audio_lanzhou_2015.xlsx


PosixPath('/Users/vishalkarda/Documents/Projects/mendx.ai/backend/data')

In [11]:
# Dataset audit: list subjects and count recordings using configured paths
# pandas is required for tabular display and analysis — fail fast if missing
import pandas as pd

RAW_DIR = DATA_DIR / DATASET_SUBDIR
print('Looking for raw data in', RAW_DIR)
if not RAW_DIR.exists():
    raise FileNotFoundError(f'Raw data directory not found: {RAW_DIR}')

subjects = []
for subj in sorted([p for p in RAW_DIR.iterdir() if p.is_dir()]):
    wavs = [p for p in subj.rglob('*.wav')]
    subjects.append({'subject': subj.name, 'n_recordings': len(wavs), 'path': str(subj), 'example_recording': str(wavs[0]) if wavs else None})

if subjects:
    df = pd.DataFrame(subjects).sort_values('subject')
    display(df)
    print('Total subjects:', len(df))
    print('Total recordings:', int(df['n_recordings'].sum()))
else:
    print('Subjects found:', 0)
    print('Total recordings:', 0)

# Read metadata workbook if present
meta_path = DATA_DIR / METADATA_WORKBOOK
if meta_path.exists():
    print('Found metadata workbook at', meta_path)
    if pd is not None:
        try:
            meta = pd.read_excel(meta_path)
            display(meta.head())
            print('Metadata rows:', len(meta))
        except Exception as e:
            print('Failed to read metadata workbook:', e)
else:
    print('No metadata workbook found at expected location:', meta_path)

Looking for raw data in /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/audio_lanzhou_2015


,subject,n_recordings,path,example_recording
0,02010001,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
1,02010002,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
2,02010003,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
3,02010004,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
4,02010005,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
5,02010006,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
6,02010008,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
7,02010009,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
8,02010010,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...
9,02010011,29,/Users/vishalkarda/Documents/Projects/mendx.ai...,/Users/vishalkarda/Documents/Projects/mendx.ai...


Total subjects: 52
Total recordings: 1508
No metadata workbook found at expected location: /Users/vishalkarda/Documents/Projects/mendx.ai/backend/data/subjects_information_audio_lanzhou_2015.xlsx
